Name: Jess Choi  
Github Username: yulimjchoi  
USC ID: 9452440573  

# 1. Time Series Classification Part 1

## (a) Download the data

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

ACTIVITIES = ["bending1", "bending2", "cycling", "lying", "sitting", "standing", "walking"]
DATA_DIR = Path("../data/AReM")

csv_files = list(DATA_DIR.glob("*/*.csv"))
sample = pd.read_csv(DATA_DIR/"bending1/dataset1.csv", comment="#", header=None)

print("Activities:", ACTIVITIES)
print("Number of Instances:", len(csv_files))
print("Number of Consecutive Values and Time Series:", sample.iloc[:, 1:].shape)

Activities: ['bending1', 'bending2', 'cycling', 'lying', 'sitting', 'standing', 'walking']
Number of Instances: 88
Number of Consecutive Values and Time Series: (480, 6)


The dataset is already downloaded, so this cell only sets its relative path and confirms the seven activity folders and the number of instances, time series, and consecutive values.

## (b) Divide the data

In [2]:
train_files = []
test_files = []

for activity in ACTIVITIES:
    files = list((DATA_DIR / activity).glob("*.csv"))

    if activity in ["bending1", "bending2"]:
        test_names = ["dataset1.csv", "dataset2.csv"]
    else:
        test_names = ["dataset1.csv", "dataset2.csv", "dataset3.csv"]

    for file in files:
        if file.name in test_names:
            test_files.append(file)
        else:
            train_files.append(file)

print("Number of training files:", len(train_files))
print("Number of test files:", len(test_files))

Number of training files: 69
Number of test files: 19


## (c) Feature Extraction

### (i) Common time-domain features
In time series classification, following features are commonly used: minimum, maximum, mean, median, standard deviation, variance, first quartile, third quartile, and range.

### (ii) Extract min, max, mean, median, sd, 1q, 3q

In [3]:
# Check data
column_names = [
    "time",
    "avg_rss12",
    "var_rss12",
    "avg_rss13",
    "var_rss13",
    "avg_rss23",
    "var_rss23"
]

all_data = {}

for file in csv_files:
    df = pd.read_csv(
        file,
        comment="#",
        header=None,
        names=column_names,
        sep=r"[,\s]+",
        engine="python",
        usecols=range(7)
    )

    all_data[file] = df

    if df.shape != (480, 7) or df.isnull().sum().sum() > 0:
        print(
            file,
            "shape:", df.shape,
            "null:", df.isnull().sum().sum()
        )

../data/AReM/sitting/dataset8.csv shape: (479, 7) null: 0


In [4]:
# Missing value handling
expected_time = np.arange(0, 120000, 250)

for file, df in all_data.items():
    if df.shape[0] != 480:
        df = df.set_index("time")
        df = df.reindex(expected_time)
        df = df.interpolate()
        df = df.reset_index()
        df = df.rename(columns={"index": "time"})

        all_data[file] = df

In [5]:
for file, df in all_data.items():
    if df.shape != (480, 7) or df.isnull().sum().sum() > 0:
        print(file, df.shape, df.isnull().sum().sum())

In [6]:
# Extract features
feature_rows = []
instance_names = []

for file, df in all_data.items():
    time_series = df.iloc[:, 1:]
    features = {}

    for i in range(6):
        series = time_series.iloc[:, i]

        features[f"min{i + 1}"] = series.min()
        features[f"max{i + 1}"] = series.max()
        features[f"mean{i + 1}"] = series.mean()
        features[f"median{i + 1}"] = series.median()
        features[f"std{i + 1}"] = series.std()
        features[f"1st_quart{i + 1}"] = series.quantile(0.25)
        features[f"3rd_quart{i + 1}"] = series.quantile(0.75)

    feature_rows.append(features)
    instance_names.append(f"{file.parent.name}/{file.name}")

In [7]:
# Create a DataFrame
feature_df = pd.DataFrame(
    feature_rows,
    index=instance_names
)

feature_df.index.name = "instance"

print(feature_df.shape)
display(feature_df.head())

(88, 42)


,min1,max1,mean1,median1,std1,1st_quart1,3rd_quart1,min2,max2,mean2,...,std5,1st_quart5,3rd_quart5,min6,max6,mean6,median6,std6,1st_quart6,3rd_quart6
instance,,,,,,,,,,,,,,,,,,,,,
bending1/dataset7.csv,36.25,48.00,43.969125,44.50,1.618364,43.31,44.67,0.0,1.50,0.413125,...,3.318301,20.5000,23.75,0.0,2.96,0.555312,0.49,0.487826,0.00,0.83
bending1/dataset6.csv,37.00,48.00,43.454958,43.25,1.386098,42.50,45.00,0.0,1.58,0.378083,...,2.488862,22.2500,24.00,0.0,5.26,0.679646,0.50,0.622534,0.43,0.87
bending1/dataset4.csv,33.00,47.75,42.179812,43.50,3.670666,39.15,45.00,0.0,3.00,0.696042,...,3.849448,30.4575,36.33,0.0,2.18,0.613521,0.50,0.524317,0.00,1.00
bending1/dataset5.csv,33.00,45.75,41.678063,41.75,2.243490,41.33,42.75,0.0,2.83,0.535979,...,2.411026,28.4575,31.25,0.0,1.79,0.383292,0.43,0.389164,0.00,0.50
bending1/dataset1.csv,37.25,45.00,40.624792,40.50,1.476967,39.25,42.00,0.0,1.30,0.358604,...,2.188449,33.0000,36.00,0.0,1.92,0.570583,0.43,0.582915,0.00,1.30


### (iii) Estimate the standard deviation

In [8]:
from scipy.stats import bootstrap

feature_std = feature_df.std()

def sample_std(data, axis=-1):
    return np.std(data, axis=axis, ddof=1)

results = []

for feature in feature_df.columns:
    values = feature_df[feature].to_numpy()

    bootstrap_result = bootstrap(
        data=(values,),
        statistic=sample_std,
        confidence_level=0.90,
        n_resamples=1000,
        method="percentile",
        random_state=42
    )

    results.append({
        "Feature": feature,
        "Standard Deviation": feature_std[feature],
        "90% CI Lower": bootstrap_result.confidence_interval.low,
        "90% CI Upper": bootstrap_result.confidence_interval.high
    })

std_result = pd.DataFrame(results)
std_result = std_result.set_index("Feature")
std_result

,Standard Deviation,90% CI Lower,90% CI Upper
Feature,,,
min1,9.569975,8.251751,10.674637
max1,4.394362,3.359479,5.279232
mean1,5.335703,4.720374,5.881827
median1,5.440054,4.801296,5.999633
std1,1.772155,1.570139,1.935291
1st_quart1,6.153590,5.558975,6.640665
3rd_quart1,5.138925,4.346374,5.856029
min2,0.000000,0.000000,0.000000
max2,5.062729,4.625258,5.411716


### (iv) Three most important time-domain features
I selected min, max, and mean. Mean represents the overall level of the signal, while the min and max represent its lower and upper extremes. These three features provide a basic description of the tendency and range of the data.

# 2. ISLR 3.7.4

(a) We would expect the training RSS of the cubic regression to be lower than or equal to that of the linear regression. The cubic model contains the linear model as a special case when the coefficients of X^2 and X^3 are 0. Therefore, the additional predictors cannot increase the training RSS. Also, the cubic model generally has lower training RSS.

(b) We would expect the linear regression test RSS is lower. Since the true relationship is linear, quadratic and cubic terms don't represent the true relationship. They may fit noise in the training data and increase the variance of the model.

(c) We would expect the cubic regression train RSS is lower. The cubic model is more flexible and it includes the linear model as a special case. Therefore, its training RSS must be less than or equal to that of the linear model.

(d) We need more information to determine the answer. If the true relationship is strongly nonlinear, the cubic model may perform better. However, if it is only slightly nonlinear, the linear model may perform better because the cubic model could overfit the data.